# SmartHire AI
## Resume Screening & Job Matching System

### Project Objective

SmartHire AI is an AI-powered Applicant Tracking System (ATS) that analyzes resumes against job descriptions and provides:

- Resume Match Score
- Candidate Ranking
- Skill Matching
- Missing Skill Detection
- Hiring Recommendations

### Tech Stack

- Python
- Pandas
- NumPy
- NLP
- Sentence Transformers
- Scikit-Learn
- FastAPI
- Streamlit

### Workflow

1. Load Resume Dataset
2. Load Job Dataset
3. Preprocess Text
4. Extract Skills
5. Generate Embeddings
6. Calculate Similarity
7. Rank Candidates
8. Deploy with FastAPI + Streamlit

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn sentence-transformers pymupdf

  Using cached pymupdf-1.28.0-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 26.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

import re
import string

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", None)

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving job_postings.csv to job_postings.csv
Saving UpdatedResumeDataSet.csv to UpdatedResumeDataSet.csv


In [ ]:
resume_df = pd.read_csv("UpdatedResumeDataSet.csv")

job_df = pd.read_csv("job_postings.csv")

In [ ]:
resume_df.shape

(962, 2)

In [ ]:
job_df.shape

(123849, 31)

In [ ]:
resume_df.head()

Category  \
0  Data Science   
1  Data Science   
2  Data Science   
3  Data Science   
4  Data Science   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [ ]:
resume_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 962 entries, 0 to 961
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  962 non-null    object
 1   Resume    962 non-null    object
dtypes: object(2)
memory usage: 15.2+ KB


In [ ]:
job_df.head()

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,"Job descriptionA leading real estate firm in New Jersey is seeking an administrative Marketing Coordinator with some experience in graphic design. You will be working closely with our fun, kind, ambitious members of the sales team and our dynamic executive team on a daily basis. This is an opportunity to be part of a fast-growing, highly respected real estate brokerage with a reputation for exceptional marketing and extraordinary culture of cooperation and inclusion.Who you are:You must be a well-organized, creative, proactive, positive, and most importantly, kind-hearted person. Please, be responsible, respectful, and cool-under-pressure. Please, be proficient in Adobe Creative Cloud (Indesign, Illustrator, Photoshop) and Microsoft Office Suite. Above all, have fantastic taste and be a good-hearted, fun-loving person who loves working with people and is eager to learn.Role:Our office is a fast-paced environment. You’ll work directly with a Marketing team and communicate daily with other core staff and our large team of agents. This description is a brief overview, but your skills and interests will be considered in what you work on and as the role evolves over time.Agent Assistance- Receive & Organize Marketing Requests from Agents- Track Tasks & Communicate with Marketing team & Agents on Status- Prepare print materials and signs for open houses- Submit Orders to Printers & Communicate & Track DeadlinesGraphic Design & Branding- Managing brand strategy and messaging through website, social media, videos, online advertising, print placement and events- Receive, organize, and prioritize marketing requests from agents- Fulfill agent design requests including postcards, signs, email marketing and property brochures using pre-existing templates and creating custom designs- Maintain brand assets and generic filesEvents & Community- Plan and execute events and promotions- Manage Contacts & Vendors for Event Planning & SponsorshipsOur company is committed to creating a diverse environment and is proud to be an equal opportunity employer. All qualified applicants will receive consideration for employment without regard to race, color, religion, gender, gender identity or expression, sexual orientation, national origin, genetics, disability, age, or veteran status.Job Type: Full-time\nPay: $18-20/hour\nExpected hours: 35 – 45 per week\nBenefits:Paid time offSchedule:8 hour shiftMonday to FridayExperience:Marketing: 1 year (Preferred)Graphic design: 2 years (Preferred)Work Location: In person\n",20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,"Requirements: \n\nWe are seeking a College or Graduate Student (can also be completed with school) with a focus in Planning, Architecture, Real Estate Development or Management or General Business. Must be able to work in an extremely fast paced environment and able to multitask and prioritize.",1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committed to serving clients with best practices to help them with change, improvements and better quality of life. We believe in providing a secure, supportive environment to grow as a clinician and learn how to foster longevity in the career which is part of our mission statement.\nThank you for taking the time to explore a career with us. We are excited to be a new group practice in the community. If you are looking for quality supervision as you work towards licensure and ability to serve populations while accepting a variety of insurance panels, we may be a good fit. Our supervisors are trained in EMDR and utilize a parts work perspective with a trauma lens.\nWe are actively 

In [ ]:
job_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [ ]:
resume_df.isnull().sum()

,0
Category,0
Resume,0


In [ ]:
job_df.isnull().sum()

,0
job_id,0
company_name,1719
title,0
description,7
max_salary,94056
pay_period,87776
location,0
company_id,1717
views,1689
med_salary,117569


In [ ]:
job_df[["title","description","formatted_experience_level","formatted_work_type"]].isnull().sum()

,0
title,0
description,7
formatted_experience_level,29409
formatted_work_type,0


In [ ]:
job_df = job_df.dropna(subset=["description"])

In [ ]:
job_df["formatted_experience_level"] = job_df["formatted_experience_level"].fillna("Not Specified")

In [ ]:
job_df[["title","description","formatted_experience_level","formatted_work_type"]].isnull().sum()

,0
title,0
description,0
formatted_experience_level,0
formatted_work_type,0


In [ ]:
job_df = job_df[["title","description","formatted_experience_level","formatted_work_type"]]

In [ ]:
job_df.isnull().sum()

,0
title,0
description,0
formatted_experience_level,0
formatted_work_type,0


# Text Cleaning

Before generating embeddings, we need to clean both resumes and job descriptions.

Raw text often contains:

- URLs
- Emails
- Special characters
- Extra spaces
- Unnecessary formatting

Cleaning improves text quality and helps embedding models focus on meaningful information.

In [ ]:
def clean_text(text):

    text = str(text)

    text = re.sub(r"http\S+|www\S+", " ", text)

    text = re.sub(r"\S+@\S+", " ", text)

    text = re.sub(r"[^a-zA-Z0-9 ]", " ", text)

    text = text.lower()

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
sample_resume = resume_df["Resume"].iloc[0]

print(clean_text(sample_resume)[:1000])

skills programming languages python pandas numpy scipy scikit learn matplotlib sql java javascript jquery machine learning regression svm na ve bayes knn random forest decision trees boosting techniques cluster analysis word embedding sentiment analysis natural language processing dimensionality reduction topic modelling lda nmf pca neural nets database visualizations mysql sqlserver cassandra hbase elasticsearch d3 js dc js plotly kibana matplotlib ggplot tableau others regular expression html css angular 6 logstash kafka python flask git docker computer vision open cv and understanding of deep learning education details data science assurance associate data science assurance associate ernst young llp skill details javascript exprience 24 months jquery exprience 24 months python exprience 24 monthscompany details company ernst young llp description fraud investigations and dispute services assurance technology assisted review tar technology assisted review assists in accelerating the 

In [ ]:
resume_df["clean_resume"] = resume_df["Resume"].apply(clean_text)

In [ ]:
job_df["clean_description"] = job_df["description"].apply(clean_text)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

os.makedirs("/content/drive/MyDrive/SmartHire-AI/data", exist_ok=True)

os.makedirs("/content/drive/MyDrive/SmartHire-AI/models", exist_ok=True)

os.makedirs("/content/drive/MyDrive/SmartHire-AI/outputs", exist_ok=True)

In [ ]:
resume_df.to_csv("/content/drive/MyDrive/SmartHire-AI/data/clean_resumes.csv",index=False)

job_df.to_csv("/content/drive/MyDrive/SmartHire-AI/data/clean_jobs.csv",index=False)

# TF-IDF Vectorization

Computers cannot directly understand text.

Before comparing resumes and job descriptions, we need to convert text into numbers.

TF-IDF (Term Frequency - Inverse Document Frequency) is a technique that converts text into numerical vectors while giving more importance to meaningful words.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(stop_words="english")

# Building the Vocabulary

Before calculating similarity, TF-IDF needs to learn the vocabulary from the text.

A vocabulary is simply a collection of unique words found in the dataset.

Example:

Resume:
Python SQL Machine Learning

Vocabulary:
['python', 'sql', 'machine', 'learning']

Each word will later become a numerical feature.

In [ ]:
tfidf.fit(resume_df["clean_resume"])

TfidfVectorizer(stop_words='english')

In [ ]:
len(tfidf.vocabulary_)

7321

In [ ]:
list(tfidf.vocabulary_.keys())[:20]

['skills',
 'programming',
 'languages',
 'python',
 'pandas',
 'numpy',
 'scipy',
 'scikit',
 'learn',
 'matplotlib',
 'sql',
 'java',
 'javascript',
 'jquery',
 'machine',
 'learning',
 'regression',
 'svm',
 'na',
 've']

# Converting Resumes into Vectors

After learning the vocabulary, TF-IDF can transform resumes into numerical vectors.

Each resume becomes a collection of weighted features based on the importance of its words.

These vectors will later be used to calculate similarity with job descriptions.

In [ ]:
resume_vectors = tfidf.transform(resume_df["clean_resume"])

In [ ]:
import os

os.makedirs("/content/drive/MyDrive/SmartHire-AI/models", exist_ok=True)

In [ ]:
import joblib

joblib.dump(tfidf, "/content/drive/MyDrive/SmartHire-AI/models/tfidf.pkl")

joblib.dump(resume_vectors, "/content/drive/MyDrive/SmartHire-AI/models/resume_vectors.pkl")

['/content/drive/MyDrive/SmartHire-AI/models/resume_vectors.pkl']

In [ ]:
import os

os.listdir("/content/drive/MyDrive/SmartHire-AI/models")

['tfidf.pkl', 'resume_vectors.pkl']

In [ ]:
import joblib

tfidf_test = joblib.load("/content/drive/MyDrive/SmartHire-AI/models/tfidf.pkl")

resume_vectors_test = joblib.load("/content/drive/MyDrive/SmartHire-AI/models/resume_vectors.pkl")

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import joblib

resume_df = pd.read_csv("/content/drive/MyDrive/SmartHire-AI/data/clean_resumes.csv")

job_df = pd.read_csv("/content/drive/MyDrive/SmartHire-AI/data/clean_jobs.csv")

tfidf = joblib.load("/content/drive/MyDrive/SmartHire-AI/models/tfidf.pkl")

resume_vectors = joblib.load("/content/drive/MyDrive/SmartHire-AI/models/resume_vectors.pkl")

In [5]:
job_df["clean_description"].isna().sum()

np.int64(7)

In [6]:
job_df = job_df.dropna(subset=["clean_description"])

In [7]:
job_df["clean_description"].isna().sum()

np.int64(0)

# Creating Job Vectors

The same TF-IDF model used for resumes will now transform job descriptions into vectors.

Using a shared vocabulary ensures that resumes and job descriptions can be compared in the same feature space.

This is a requirement before calculating cosine similarity.

In [8]:
job_vectors = tfidf.transform(job_df["clean_description"])

In [9]:
job_vectors


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 22130026 stored elements and shape (123835, 7321)>

# Understanding Vector Dimensions

Before calculating similarity, it is important to verify that resume vectors and job vectors have the same dimensions.

Since both were created using the same TF-IDF vocabulary, they should exist in the same feature space.

This ensures that cosine similarity can be calculated correctly.

In [10]:
resume_vectors.shape, job_vectors.shape

((962, 7321), (123835, 7321))

In [11]:
job_text = job_df["clean_description"].iloc[0]

# Converting the Job Description into a Vector

The TF-IDF model has already learned the vocabulary from resumes.

Now the selected job description is transformed into the same vector space.

This allows direct comparison between the job and all resumes using cosine similarity.

In [12]:
job_vector = tfidf.transform([job_text])

In [13]:
job_vector.shape

(1, 7321)

# Calculating Resume Similarity Scores

Cosine Similarity measures how similar two vectors are.

In SmartHire AI, it is used to compare the job description vector with every resume vector.

Higher scores indicate stronger alignment between candidate skills and job requirements.

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(job_vector, resume_vectors)

In [15]:
similarity_scores.shape

(1, 962)

# Ranking Candidates

After calculating similarity scores, candidates are ranked from highest to lowest score.

The highest-ranked candidates are considered the best matches for the job description.

In [16]:
#Converting 2D into 1D [Before: (1,962)] [After:(962,)]

resume_df["match_score"] = similarity_scores.flatten()

# Ranking Candidates by Match Score

After calculating similarity scores, candidates are sorted from highest to lowest match percentage.

This ranking helps recruiters quickly identify the most relevant candidates for a job opening.

In [17]:
top_candidates = resume_df.sort_values("match_score", ascending=False).head(10)

# Reviewing Top Ranked Candidates

The ranking model has identified the candidates with the highest similarity scores.

The next step is to inspect these results and determine whether the rankings are meaningful and aligned with the job requirements.

In [18]:
top_candidates[["Category", "match_score"]]

,Category,match_score
247,Sales,0.156574
242,Sales,0.156574
237,Sales,0.156574
232,Sales,0.156574
227,Sales,0.156574
262,Sales,0.156574
257,Sales,0.156574
252,Sales,0.156574
118,Arts,0.119144
106,Arts,0.119144


# Sample Job Description

A Data Scientist role requiring:

- Python
- SQL
- Machine Learning
- Pandas
- Data Analysis
# Simulating a Real Recruiter Job Description

In the final Streamlit application, recruiters will paste a job description.

To replicate the production workflow, we will create a sample Data Scientist job description and rank all resumes against it.

In [19]:
job_text = """
Data Scientist

Required Skills:
Python
SQL
Machine Learning
Pandas
Data Analysis
Scikit Learn
Data Visualization
"""

In [20]:
job_vector = tfidf.transform([job_text])

# Calculating Similarity Against All Resumes

The transformed job description is compared with every resume vector using cosine similarity.

The resulting score indicates how closely a resume aligns with the job requirements.

In [21]:
similarity_scores = cosine_similarity(job_vector, resume_vectors)

In [22]:
resume_df["match_score"] = similarity_scores.flatten()

In [23]:
resume_df.head()

,Category,Resume,clean_resume,match_score
0,Data Science,Skills * Programming Languages: Python (pandas...,skills programming languages python pandas num...,0.199571
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...,education details may 2013 to may 2017 b e uit...,0.148398
2,Data Science,"Areas of Interest Deep Learning, Control Syste...",areas of interest deep learning control system...,0.128590
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...,skills r python sap hana tableau sap hana sql ...,0.126180
4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab...",education details mca ymcaust faridabad haryan...,0.136914


In [24]:
top_candidates = resume_df.sort_values("match_score", ascending=False).head(10)

In [25]:
top_candidates[["Category", "match_score"]]

,Category,match_score
26,Data Science,0.396677
6,Data Science,0.396677
36,Data Science,0.396677
16,Data Science,0.396677
29,Data Science,0.318943
9,Data Science,0.318943
19,Data Science,0.318943
39,Data Science,0.318943
8,Data Science,0.267967
38,Data Science,0.267967


In [27]:
top_candidates["match_percentage"] = (top_candidates["match_score"] * 100).round(2)

In [29]:
top_candidates[["Category", "match_score","match_percentage"]]

,Category,match_score,match_percentage
26,Data Science,0.396677,39.67
6,Data Science,0.396677,39.67
36,Data Science,0.396677,39.67
16,Data Science,0.396677,39.67
29,Data Science,0.318943,31.89
9,Data Science,0.318943,31.89
19,Data Science,0.318943,31.89
39,Data Science,0.318943,31.89
8,Data Science,0.267967,26.80
38,Data Science,0.267967,26.80


In [30]:
resume_df["match_score"].describe()

,match_score
count,962.000000
mean,0.036278
std,0.052445
min,0.000000
25%,0.006697
50%,0.019943
75%,0.045756
max,0.396677


In [33]:
def rank_resumes(job_text, tfidf, resume_vectors, resume_df):

    job_vector = tfidf.transform([job_text])

    scores = cosine_similarity(job_vector, resume_vectors).flatten()

    results = resume_df.copy()

    results["match_score"] = scores

    results = results.sort_values("match_score", ascending=False)

    return results

# Resume Ranking Engine

The ranking engine compares a job description against all available resumes using TF-IDF vectorization and cosine similarity.

Each resume receives a similarity score, and the resumes are ranked from highest to lowest relevance.

In [34]:
results = rank_resumes(job_text, tfidf, resume_vectors, resume_df)

results[["Category","match_score"]].head(10)

,Category,match_score
26,Data Science,0.396677
6,Data Science,0.396677
36,Data Science,0.396677
16,Data Science,0.396677
29,Data Science,0.318943
9,Data Science,0.318943
19,Data Science,0.318943
39,Data Science,0.318943
8,Data Science,0.267967
38,Data Science,0.267967


In [35]:
!pip install pymupdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 69.2 MB/s eta 0:00:00


In [38]:
import fitz

def extract_text_from_pdf(pdf_path):

    text = ""

    pdf = fitz.open(pdf_path)

    for page in pdf:

        text += page.get_text()

    pdf.close()

    return text

In [40]:
def load_resumes_from_folder(folder_path):

    resumes = []

    for file in os.listdir(folder_path):

        if file.endswith(".pdf"):

            pdf_path = os.path.join(folder_path, file)

            text = extract_text_from_pdf(pdf_path)

            resumes.append({
                "file_name": file,
                "resume_text": text
            })

    return pd.DataFrame(resumes)

In [41]:
resume_df.columns

Index(['Category', 'Resume', 'clean_resume', 'match_score'], dtype='object')

# Resume Preprocessing Pipeline

After extracting text from PDF files, the same preprocessing steps used during training must be applied.

This ensures consistency between training data and uploaded resumes.

In [42]:
def preprocess_resumes(resume_df):

    resume_df["clean_resume"] = resume_df["resume_text"].apply(clean_text)

    return resume_df

# Converting Uploaded Resumes into TF-IDF Vectors

The uploaded resumes are transformed into the same TF-IDF feature space that was learned during training.

This allows direct comparison with the job description.

In [44]:
def vectorize_resumes(resume_df, tfidf):

    return tfidf.transform(resume_df["clean_resume"])

In [45]:
def analyze_resumes(folder_path, job_text, tfidf):

    resumes = load_resumes_from_folder(folder_path)

    resumes = preprocess_resumes(resumes)

    resume_vectors = vectorize_resumes(resumes, tfidf)

    job_vector = tfidf.transform([clean_text(job_text)])

    scores = cosine_similarity(job_vector, resume_vectors).flatten()

    resumes["match_score"] = (scores * 100).round(2)

    resumes = resumes.sort_values("match_score", ascending=False)

    return resumes

In [46]:
joblib.dump(tfidf, "/content/drive/MyDrive/SmartHire-AI/models/tfidf.pkl")

['/content/drive/MyDrive/SmartHire-AI/models/tfidf.pkl']

# Conclusion

In this notebook, a complete resume screening pipeline was developed using Natural Language Processing (NLP) techniques.

The workflow includes:

- Text preprocessing and normalization
- TF-IDF vectorization
- Resume representation in a numerical feature space
- Cosine similarity based candidate ranking
- PDF resume text extraction
- Multiple resume processing pipeline
- End-to-end resume analysis workflow

The resulting system can compare resumes against a job description and rank candidates based on their relevance, providing a foundation for an Applicant Tracking System (ATS).

Future development will focus on building a user interface, integrating deployment components, and extending the system with additional candidate evaluation features.